# Week 4 · Systems Thinking for DE — pre-work: bring the stack up

The Week 4 class decided the stack (ADR-000); this notebook brings it up before Week 5, when the building starts. Sections 0–3 are the pre-work (two to three hours): run them inside the Codespace first, then again locally once `make doctor` passes. Sections 4 and 5 feed the lab: the capacity model for the design doc and the ADR-001 template. Shell cells start with `!` and run from the repo root.

| Stage | What you have at the end | Proof |
| --- | --- | --- |
| 1 | Core stack is up | all three PASS lines in section 1 |
| 2 | OwoPay seeded, three systems toured | row counts, recent events and buckets shown in section 2 |
| 3 | Bridge lab: your Airflow DAG and Kafka producer run here | DAG listed, fresh events on `txn.events` |
| 4 | Capacity model filled | every number carries its assumption; `docs/design/capacity_model.md` written |
| 5 | ADR-001 drafted | `docs/adr/ADR-001-storage-topology.md` and the design-doc skeleton exist |

`make check-w4` runs the same checks headlessly; the Week 5 class opens with it. Column and topic names follow the course generator's schema (`docs/schema.md` in the repo): `transactions(txn_id, customer_id, merchant_id, agent_id, channel, amount_kobo, status, created_at)` and the topics `txn.events` and `app.events`. If a stage fails, say so in the class channel and keep going; the Day 6 setup clinic picks it up.

## 0 · Setup

Connection defaults match the course stack's `.env` (see the Conventions section of the Course Tech Stack document). They come from the repo's `.env` (copied from `.env.example`), so a port you moved there, such as `PG_PORT=5433` next to a local Postgres, applies here too.

In [ ]:
import os, sys, time, json, uuid, datetime as dt
from pathlib import Path

# The repo root is the folder that holds docker-compose.yml and the Makefile.
REPO_ROOT = Path(os.environ.get('OWOPAY_REPO', Path.cwd())).resolve()
if not (REPO_ROOT / 'docker-compose.yml').exists() and (REPO_ROOT.parent / 'docker-compose.yml').exists():
    REPO_ROOT = REPO_ROOT.parent
os.chdir(REPO_ROOT)
print('Working in', REPO_ROOT)

# One source of settings for the whole repo: .env.example, then your .env, then environment variables.
sys.path.insert(0, str(REPO_ROOT / 'generator'))
from owopay.config import SETTINGS as S

CFG = {
    'pg_host': S['PG_HOST'],
    'pg_port': int(S['PG_PORT']),
    'pg_db': S['PG_DB'],
    'pg_user': S['PG_USER'],
    'pg_password': S['PG_PASSWORD'],
    'kafka': S['KAFKA_BOOTSTRAP'],
    's3_endpoint': S['S3_ENDPOINT'],
    's3_key': S['S3_ACCESS_KEY'],
    's3_secret': S['S3_SECRET_KEY'],
    's3_region': S['S3_REGION'],
    'airflow_service': S['AIRFLOW_SERVICE'],
    'airflow_url': S['AIRFLOW_URL'],
    'console_url': 'http://localhost:8082',
    's3_browser_url': 'http://localhost:8888',
}
print(json.dumps({k: v for k, v in CFG.items() if 'secret' not in k and 'password' not in k}, indent=2))

In [ ]:
%pip install -q "psycopg[binary]" pandas confluent-kafka boto3

## 1 · Stack up — stage 1

The `core` profile starts Postgres (the OwoPay OLTP database and the empty `dw` warehouse), Redpanda with its console and schema registry, and SeaweedFS, the S3-compatible object store. Nothing else yet: one profile at a time keeps an 8 GB laptop comfortable.

In [ ]:
!make up PROFILE=core
!docker compose ps

In [ ]:
import pandas as pd

def check(name, fn, hint):
    try:
        detail = fn()
        print('PASS  %s: %s' % (name, detail))
        return True
    except Exception as e:
        print('FAIL  %s: %s: %s' % (name, type(e).__name__, e))
        print('      hint: %s' % hint)
        return False

def pg_conn(db=None):
    import psycopg
    return psycopg.connect(host=CFG['pg_host'], port=CFG['pg_port'], dbname=db or CFG['pg_db'],
                           user=CFG['pg_user'], password=CFG['pg_password'], connect_timeout=5)

def pg_ok():
    with pg_conn() as conn:
        return conn.execute('select version()').fetchone()[0].split(',')[0]

def kafka_ok():
    from confluent_kafka.admin import AdminClient
    md = AdminClient({'bootstrap.servers': CFG['kafka']}).list_topics(timeout=5)
    return '%d broker(s), topics: %s' % (len(md.brokers), sorted(md.topics))

def s3_client():
    import boto3
    from botocore.config import Config
    return boto3.client('s3', endpoint_url=CFG['s3_endpoint'], aws_access_key_id=CFG['s3_key'],
                        aws_secret_access_key=CFG['s3_secret'], region_name=CFG['s3_region'],
                        config=Config(s3={'addressing_style': 'path'}))

def s3_ok():
    names = [b['Name'] for b in s3_client().list_buckets()['Buckets']]
    return 'buckets: %s' % names

results = [
    check('Postgres', pg_ok, 'is the core profile up? run the cell above, then read docker compose ps'),
    check('Redpanda', kafka_ok, 'port 9092 must be published; see the redpanda service in docker-compose.yml'),
    check('SeaweedFS S3', s3_ok, 'port 8333 must be published; the S3 keys live in .env and seaweedfs/s3.json'),
]
print()
print('Stage 1:', 'PASS' if all(results) else 'not yet — fix the FAIL lines before moving on')

## 2 · Seed OwoPay and tour the three systems — stage 2

`make seed` wraps the generator (`python -m owopay.generate --day ... --rows ...`): deterministic per seed, so everyone's numbers match. It also creates the `bronze`, `silver` and `quarantine` buckets. Chaos flags stay off until Week 7.

In [ ]:
!make seed DAY=2026-11-03 ROWS=200000

In [ ]:
def q(conn, sql, params=None):
    cur = conn.execute(sql, params)
    return pd.DataFrame(cur.fetchall(), columns=[d.name for d in cur.description])

with pg_conn() as conn:
    tables = [r[0] for r in conn.execute("select table_name from information_schema.tables where table_schema = 'public' order by 1")]
    counts = {t: conn.execute('select count(*) from "%s"' % t).fetchone()[0] for t in tables}
    display(pd.DataFrame({'table': list(counts), 'rows': list(counts.values())}))
    if 'transactions' in tables:
        display(q(conn, 'select * from transactions order by created_at desc limit 5'))

The finance team's question, asked of the OLTP database. At 200k rows it is instant; the plan below is what runs at a million rows a day for three years, against the same tables customers are writing to. This is the "three stores" slide in one query.

In [ ]:
ANALYTICS_SQL = '''
select date_trunc('day', created_at) as day, status, count(*) as txns, sum(amount_kobo) / 100.0 as naira
from transactions
group by 1, 2
order by 1, 2
'''

with pg_conn() as conn:
    t0 = time.perf_counter()
    daily = q(conn, ANALYTICS_SQL)
    ms = (time.perf_counter() - t0) * 1000
    print('%.0f ms for a full scan over %s rows' % (ms, format(counts.get('transactions', 0), ',')))
    display(daily.head(10))
    plan = conn.execute('explain (analyze, buffers) ' + ANALYTICS_SQL).fetchall()
    print()
    print('Query plan (first lines):')
    for line in plan[:12]:
        print('  ', line[0])

In [ ]:
from confluent_kafka import Consumer, TopicPartition

def peek(topic, n=5, timeout=5.0):
    """Return up to n recent JSON events per partition of a topic, without committing offsets."""
    c = Consumer({'bootstrap.servers': CFG['kafka'], 'group.id': 'week04-peek-' + uuid.uuid4().hex[:8],
                  'auto.offset.reset': 'latest', 'enable.auto.commit': False})
    md = c.list_topics(topic, timeout=timeout)
    if topic not in md.topics or md.topics[topic].error is not None:
        c.close()
        raise RuntimeError('topic %s not found — has make seed run?' % topic)
    parts = []
    for p in md.topics[topic].partitions:
        lo, hi = c.get_watermark_offsets(TopicPartition(topic, p), timeout=timeout)
        parts.append(TopicPartition(topic, p, max(lo, hi - n)))
    c.assign(parts)
    out, deadline = [], time.time() + timeout
    while len(out) < n * len(parts) and time.time() < deadline:
        m = c.poll(0.5)
        if m is not None and m.error() is None:
            out.append(json.loads(m.value()))
    c.close()
    return out

events = peek('txn.events')
print(len(events), 'recent events on txn.events')
display(pd.DataFrame(events).head())

In [ ]:
s3 = s3_client()
for b in s3.list_buckets()['Buckets']:
    objs = s3.list_objects_v2(Bucket=b['Name'], MaxKeys=1000)
    print('%-20s %5d object(s)' % (b['Name'], objs.get('KeyCount', 0)))
print()
print('Empty buckets are expected today: Bronze and Silver start filling in Week 6.')

**Stage 2.** Open the three consoles once so you know where they live: Redpanda Console at `CFG['console_url']`, the SeaweedFS file browser at `CFG['s3_browser_url']`, and Postgres through `docker compose exec postgres psql -U owopay`. Then answer in one line each: which system holds the record of truth, which one is a queue, and which one is a filing cabinet?

## 3 · Bridge lab — your Airflow DAG and Kafka producer in this stack — stage 3

You already wrote a DAG and a producer earlier in the programme. Both run here unchanged except for connection settings.

1. Copy your DAG file into `airflow/dags/`. Airflow picks it up within a minute; the cell below lists what it sees.
2. Copy your producer into `ingestion/bridge/producer.py`, point its bootstrap server at `CFG['kafka']` and its topic at `txn.events`, and run it from a terminal.
3. Verify both with the cells below. If adapting your producer takes more than ten minutes, run the reference producer and move on; the Day 6 clinic will finish the port.

In [ ]:
!make up PROFILE=orchestration
!docker compose exec -T {CFG['airflow_service']} airflow dags list

In [ ]:
DAG_ID = 'your_dag_id'   # the dag_id from your DAG file

!docker compose exec -T {CFG['airflow_service']} airflow dags unpause {DAG_ID}
!docker compose exec -T {CFG['airflow_service']} airflow dags trigger {DAG_ID}
print('Watch the run at', CFG['airflow_url'])

In [ ]:
from confluent_kafka import Producer

def reference_producer(n=10, topic='txn.events'):
    """Fallback producer: ten well-formed events so the rest of the pre-work does not wait on a port."""
    p = Producer({'bootstrap.servers': CFG['kafka']})
    now = dt.datetime.now(dt.timezone.utc)
    for i in range(n):
        evt = {
            'event_id': str(uuid.uuid4()),
            'txn_id': str(uuid.uuid4()),
            'event_type': 'initiated',
            'amount_kobo': 150000 + 100 * i,
            'channel': 'ussd' if i % 2 else 'app',
            'occurred_at': (now + dt.timedelta(seconds=i)).isoformat(),
            'producer': 'week04-reference',
        }
        p.produce(topic, key=evt['txn_id'], value=json.dumps(evt))
    p.flush(10)
    return n

# Uncomment to use the fallback:
# print(reference_producer(), 'events produced')

In [ ]:
recent = peek('txn.events', n=50)
cutoff = (dt.datetime.now(dt.timezone.utc) - dt.timedelta(minutes=10)).isoformat()
fresh = [e for e in recent if str(e.get('occurred_at', e.get('created_at', ''))) >= cutoff]
print(len(fresh), 'events from the last 10 minutes out of', len(recent), 'seen')
print('Stage 3:', 'PASS' if fresh else 'no fresh events — is your producer pointing at %s and topic txn.events?' % CFG['kafka'])

## 4 · Capacity model — stage 4

The method from the class, as arithmetic you can argue with:

1. events per day ÷ 86,400 → events per second; × peak multiplier
2. × bytes per event → raw GB per day
3. × days kept, compounded by growth → GB stored; ÷ Parquet ratio once it is columnar
4. one partition per day unless a day exceeds about a gigabyte
5. price it: storage is cents per GB-month, compute is dollars per hour, egress is the trap

Every input below carries a note saying where it came from. Change an input, rerun, and watch every result move with it. The unit prices are placeholders: replace them with the current list prices of the provider you would actually use.

In [ ]:
ASSUMPTIONS = {
    # name: (value, where it comes from)
    'transactions_per_day':      (1_000_000, 'OwoPay today, from the business'),
    'events_per_transaction':    (3,         'initiated, authorised, then settled or failed'),
    'bytes_per_event':           (1_024,     'JSON on the wire; measure yours from the peek above'),
    'snapshot_rows_per_day':     (1_000_000, 'OLTP rows touched per day, about one per transaction'),
    'bytes_per_snapshot_row':    (300,       'measure with pg_column_size on a transactions row'),
    'growth_per_year':           (0.20,      'management forecast; try 0.5 to see the sensitivity'),
    'retention_years':           (3,         'the regulator wants three years online'),
    'peak_multiplier':           (10,        'rule of thumb for salary days and promotions'),
    'parquet_ratio':             (7,         'raw JSON to Parquet; 5 to 10 is typical for events'),
    'storage_usd_per_gb_month':  (0.023,     'placeholder for object storage, standard tier — check the provider page'),
    'warehouse_usd_per_gb_month': (0.10,     'placeholder for managed warehouse storage — check the provider page'),
}
A = {k: v[0] for k, v in ASSUMPTIONS.items()}

SEC_PER_DAY = 86_400
GB = 1024 ** 3
rows = []

def add(name, value, how):
    rows.append((name, value, how))

events_per_day = A['transactions_per_day'] * A['events_per_transaction']
eps = events_per_day / SEC_PER_DAY
add('Events per day', format(events_per_day, ',.0f'), 'transactions × events per transaction')
add('Events per second, average', format(eps, ',.1f'), 'events per day ÷ 86,400')
add('Events per second, peak', format(eps * A['peak_multiplier'], ',.0f'), 'average × peak multiplier')

raw_event_gb_day = events_per_day * A['bytes_per_event'] / GB
raw_snap_gb_day = A['snapshot_rows_per_day'] * A['bytes_per_snapshot_row'] / GB
raw_gb_day = raw_event_gb_day + raw_snap_gb_day
add('Raw GB per day, events', format(raw_event_gb_day, ',.2f'), 'events per day × bytes per event')
add('Raw GB per day, snapshots', format(raw_snap_gb_day, ',.2f'), 'snapshot rows × bytes per row')

years = A['retention_years']
daily_by_year = [raw_gb_day * (1 + A['growth_per_year']) ** y for y in range(years)]
raw_total_gb = sum(d * 365 for d in daily_by_year)
parquet_total_gb = raw_total_gb / A['parquet_ratio']
add('Raw GB over %d years' % years, format(raw_total_gb, ',.0f'), 'daily GB × 365, compounded by growth each year')
add('Compressed GB in the lake', format(parquet_total_gb, ',.0f'), 'raw ÷ Parquet ratio')
add('Partitions per table', format(years * 365, ','), 'one per day; each day is well under 1 GB at this volume')
add('Lake storage, USD per month at year %d' % years, format(parquet_total_gb * A['storage_usd_per_gb_month'], ',.0f'), 'compressed GB × object storage price')
add('Warehouse storage, USD per month', format(0.1 * parquet_total_gb * A['warehouse_usd_per_gb_month'], ',.0f'), 'assumes Gold marts are a tenth of the lake')

capacity = pd.DataFrame(rows, columns=['Quantity', 'Value', 'How it was computed'])
display(capacity)

In [ ]:
def compressed_gb(retention_years, growth):
    return sum(raw_gb_day * (1 + growth) ** y * 365 for y in range(retention_years)) / A['parquet_ratio']

sens = pd.DataFrame({'growth %d%%' % round(g * 100): [round(compressed_gb(r, g)) for r in (1, 3, 5)] for g in (0.1, 0.2, 0.5)},
                    index=['1 year', '3 years', '5 years'])
sens.index.name = 'compressed GB kept'
display(sens)

out = REPO_ROOT / 'docs' / 'design' / 'capacity_model.md'
out.parent.mkdir(parents=True, exist_ok=True)
lines = ['# Capacity model — OwoPay', '', '## Assumptions', '', '| Input | Value | Where it comes from |', '| --- | --- | --- |']
lines += ['| %s | %s | %s |' % (k, v[0], v[1]) for k, v in ASSUMPTIONS.items()]
lines += ['', '## Results', '', '| Quantity | Value | How it was computed |', '| --- | --- | --- |']
lines += ['| %s | %s | %s |' % r for r in rows]
out.write_text('\n'.join(lines) + '\n')
print('Written', out.relative_to(REPO_ROOT), '— link it from section 3 of the design doc')

**Stage 4.** Every number in the table has its assumption beside it. Two questions for your pair: at what growth rate does keeping three years in a warehouse alone stop being affordable, and which single input are you least sure of?

## 5 · ADR-001 and the System Design Doc v0 — stage 5

An Architecture Decision Record is one page, kept forever: context, the options considered (including the obvious one), the decision in one sentence, and the consequences. ADR-000 (the course stack) was ratified in class; ADR-001 is yours: where does OwoPay's data live — warehouse only, lake only, or lake plus warehouse?

The cell below creates the two files for the lab from templates if they do not exist yet. It never overwrites a file you have started.

In [ ]:
ADR = '''# ADR-001: Where OwoPay's data lives

- Status: proposed
- Date: {date}
- Deciders: {author}

## Context
Which requirement forces this choice? Cite the four questions and the SLOs from the design doc, and quote the capacity model: raw GB per day, GB kept, peak events per second.

## Options considered
1. Warehouse only —
2. Lake only —
3. Lake plus warehouse —

For each: what it does well for OwoPay, what it costs, what it makes hard.

## Decision
One sentence.

## Consequences
What becomes easier, what becomes harder, what you give up, what you must now monitor.
'''

DESIGN = '''# OwoPay analytics platform — System Design Doc v0

Author: {author} · Date: {date} · Status: draft

## 1. The four questions
- Volume:
- Velocity:
- Variety:
- Value (who consumes it, which decision it feeds):

## 2. Service level objectives
| Consumer | Freshness | Availability | Correctness |
| --- | --- | --- | --- |
| Finance dashboard | | | |
| Regulator T+1 report | | | |
| Fraud signal | | | |

## 3. Capacity model
See docs/design/capacity_model.md, generated from the Week 4 notebook. State the one input you are least sure of.

## 4. Context diagram
Who uses the platform and which systems feed it. Mermaid or an image under docs/design/.

## 5. Container diagram
What runs where: sources, ingestion, lake, transform, warehouse, serving, orchestration. The boxes must match the stack you brought up.

## 6. Decisions
- ADR-000 course stack: docs/adr/ADR-000-course-stack.md (ratified in class)
- ADR-001 storage topology: docs/adr/ADR-001-storage-topology.md

## 7. Nigerian constraints and how the design answers them
| Constraint | Design response |
| --- | --- |
| Intermittent connectivity | |
| Naira revenue, dollar-priced cloud | |
| Two-person team | |
| Power cuts and shared laptops | |
| Regulators and auditors | |

## 8. Open questions
-
'''

author = os.environ.get('GIT_AUTHOR_NAME') or os.environ.get('USER') or 'your name'
today = dt.date.today().isoformat()
targets = [
    (REPO_ROOT / 'docs' / 'adr' / 'ADR-001-storage-topology.md', ADR),
    (REPO_ROOT / 'docs' / 'design' / 'system-design-v0.md', DESIGN),
]
for path, text in targets:
    path.parent.mkdir(parents=True, exist_ok=True)
    if path.exists():
        print('exists, left unchanged:', path.relative_to(REPO_ROOT))
    else:
        path.write_text(text.format(author=author, date=today))
        print('created:', path.relative_to(REPO_ROOT))

**Stage 5.** Fill the Context section of ADR-001 now, while the class discussion is fresh; the options and consequences are the lab.

### What remains for the lab (due Day 7, 23:59)

- Complete `docs/design/system-design-v0.md` and `docs/adr/ADR-001-storage-topology.md`
- Make `make demo` print live row counts from Postgres, the number of events on `txn.events`, and the buckets in SeaweedFS
- Install Docker locally and pass `make doctor`, or write in the PR why you are staying in Codespaces
- Open the pull request using the repo template; CI is not switched on until Week 8, so a reviewer runs `make up && make demo` from a clean clone

Accepted when it runs from a clean clone or a fresh Codespace, every number in the capacity model has its assumption beside it, the ADR names the options that lost, and the container diagram matches the stack you brought up.

In [ ]:
# Before you leave: the three targets a reviewer will run.
!make check-w4
!make demo
!make doctor